# 弱智吧AI · ruozhi

参考 [karpathy/nanochat](https://github.com/karpathy/nanochat)，在 Colab 上从零训练一个会讲弱智吧金句的小语言模型：

1. **数据**：GitHub / HuggingFace 上开源的百度弱智吧数据集 → SFT 对话数据
2. **分词器**：在中文网页 + 弱智吧语料上训练 16k 词表的 byte-level BPE
3. **预训练**：FineWeb-2 中文（通用网页）+ 少量弱智吧原文
4. **SFT**：弱智吧问答 / 金句 / 续写
5. **聊天**：命令行或 Gradio 网页（可生成公开链接）

`运行时 → 更改运行时类型 → GPU`。T4 上 d6（2300 万参数）全流程约 1.5 小时；A100 约 20 分钟。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 获取代码

In [ ]:
REPO = "https://github.com/ubisoft-potato/ruozhi.git"
BRANCH = "main"  # @param {type:"string"}
!rm -rf /content/ruozhi && git clone -q -b {BRANCH} {REPO} /content/ruozhi
%cd /content/ruozhi
!pip install -q tokenizers datasets gradio

## 2. （可选）用 Google Drive 保存进度

Colab 断线会清空 `/content`。打开 `USE_DRIVE` 后，训练前会从 Drive 恢复已有的数据 / 分词器 / 检查点，最后一格会把结果备份回 Drive。
训练本身读写本地磁盘（直接读 Drive 上的 memmap 很慢）。

In [ ]:
USE_DRIVE = False  # @param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/ruozhi_artifacts"
import os
os.environ["RUOZHI_BASE_DIR"] = "/content/ruozhi/artifacts"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists(DRIVE_DIR):
        !mkdir -p /content/ruozhi/artifacts && rsync -a {DRIVE_DIR}/ /content/ruozhi/artifacts/
        !ls -R /content/ruozhi/artifacts | head -40

## 3. 弱智吧数据 → SFT 对话

In [ ]:
!python -m scripts.prepare_ruozhiba
!head -n 5 artifacts/sft/train.jsonl

## 4. 预训练数据 + 分词器

从 HuggingFace 流式读取 FineWeb-2 中文（`cmn_Hani`），训练分词器并把文本编码成 `uint16` token 文件。
3 亿 token 在 Colab 上约 20–30 分钟（瓶颈是 CPU 分词）。只想快速跑通可以把 `MAX_TOKENS` 改成 `30_000_000`。

In [ ]:
MAX_TOKENS = 300_000_000  # @param {type:"integer"}
DATASET = "fineweb2"  # @param ["fineweb2", "fineweb-edu-zh", "wiki"]
!python -m scripts.prepare_pretrain --max_tokens {MAX_TOKENS} --dataset {DATASET}

## 5. 预训练

`DEPTH` 是唯一的模型大小旋钮：`n_embd = 64 × depth`。默认按 Chinchilla 比例训练 `20 × 参数量` 个 token。

| depth | 参数量 | 训练 token | T4 | A100 |
|---|---|---|---|---|
| 4 | 11.5M | 230M | ~15 min | ~3 min |
| 6 | 23.2M | 464M | ~1 h | ~8 min |
| 8 | 42.0M | 840M | ~3 h | ~25 min |

（耗时为粗略估计。）断线后加 `--resume` 可从最近的检查点继续。

In [ ]:
DEPTH = 6  # @param {type:"integer"}
!python -m scripts.base_train --depth {DEPTH}

## 6. SFT：学习弱智吧

In [ ]:
!python -m scripts.sft_train --depth {DEPTH}

## 7. 试一试

In [ ]:
for p in ["来一条弱智吧金句", "只剩一个心脏了还能活吗？", "为什么我爸妈结婚的时候没有邀请我？", "你是谁？"]:
    print("你>", p)
    !python -m scripts.chat_cli --depth {DEPTH} -p "{p}"
    print()

网页版（`--share` 会给出一个 72 小时有效的公开链接，可以发给朋友）。停止运行这一格即关闭。

In [ ]:
!python -m scripts.chat_web --depth {DEPTH} --share

## 8. 备份到 Drive

In [ ]:
if USE_DRIVE:
    !mkdir -p {DRIVE_DIR} && rsync -a --exclude 'pretrain/*.bin' /content/ruozhi/artifacts/ {DRIVE_DIR}/
    # token 文件较大（~600MB），需要的话去掉 --exclude

## 9. （进阶）Scaling law 实验

对每个计算量预算 C、每个 depth 都用恰好 C FLOPs 训练，画出 IsoFLOP 曲线，拟合最优参数量 N\*(C) 和 token 数 D\*(C)，用来决定下一步把模型扩到多大。
T4 上默认配置约 1.5 小时。

In [ ]:
!python -m scripts.scaling_laws --budgets 1e15 3e15 1e16 --depths 2 3 4 5 6 8 --extra_args "--eval_tokens 524288"
from IPython.display import Image
Image("artifacts/scaling/scaling_laws.png")